# Tech Challenge - Obesity

## Previsão nível de obesidade

In [1]:
%pip install pandas
import pandas as pd

Note: you may need to restart the kernel to use updated packages.


In [2]:
dataset = pd.read_csv('https://raw.githubusercontent.com/Malbuquerque-data/fase-4-fiap/refs/heads/main/Obesity.csv')

In [4]:
dataset.head()

,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [13]:
dataset['family_history'] = dataset['family_history'].replace(['yes','no'],[1,0])
dataset['FAVC'] = dataset['FAVC'].replace(['yes','no'],[1,0])
dataset['SMOKE'] = dataset['SMOKE'].replace(['yes','no'],[1,0])
dataset['SCC'] = dataset['SCC'].replace(['yes','no'],[1,0])
dataset['CAEC'] = dataset['CAEC'].replace(['no','Sometimes','Frequently','Always'],[0,1,2,3])
dataset['CALC'] = dataset['CALC'].replace(['no','Sometimes','Frequently','Always'],[0,1,2,3])

### Feature Engineering

- BMI = Weight / Height²
- Variáveis com números decimais serão arredondadas  (FCVC, NCP, CH2O, FAF, TUE)

In [25]:
df = dataset.copy()
df['BMI'] = df['Weight']/(df['Height']**2)
ord_cols = ['FCVC','NCP','CH2O','FAF','TUE']
for c in ord_cols:
    if c in df.columns:
        df[c] = df[c].round().astype(int)

df[ord_cols + ['BMI']].describe()

,FCVC,NCP,CH2O,FAF,TUE,BMI
count,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000
mean,2.423496,2.687826,2.014685,1.006632,0.664614,29.700159
std,0.583905,0.809680,0.688616,0.895462,0.674009,8.011337
min,1.000000,1.000000,1.000000,0.000000,0.000000,12.998685
25%,2.000000,3.000000,2.000000,0.000000,0.000000,24.325802
50%,2.000000,3.000000,2.000000,1.000000,1.000000,28.719089
75%,3.000000,3.000000,2.000000,2.000000,1.000000,36.016501
max,3.000000,4.000000,3.000000,3.000000,2.000000,50.811753


### Pipeline

In [24]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder
from imblearn.over_sampling import SMOTE

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

TARGET = 'Obesity'
X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols)
])

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample'
)

clf = Pipeline([
    ('preprocess', preprocess),
    ('model', model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

C:\Users\mayco\AppData\Local\Temp\ipykernel_17636\1763581681.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


0.9763593380614657

In [29]:
#Avaliando o modelo
labels = sorted(y.unique())
cm = confusion_matrix(y_test, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df

print(classification_report(y_test, pred))


                     precision    recall  f1-score   support

Insufficient_Weight       1.00      0.98      0.99        54
      Normal_Weight       0.89      0.98      0.93        58
     Obesity_Type_I       0.99      1.00      0.99        70
    Obesity_Type_II       0.98      0.98      0.98        60
   Obesity_Type_III       1.00      0.98      0.99        65
 Overweight_Level_I       0.98      0.91      0.95        58
Overweight_Level_II       1.00      0.98      0.99        58

           accuracy                           0.98       423
          macro avg       0.98      0.98      0.98       423
       weighted avg       0.98      0.98      0.98       423

